In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('maxcleaned.csv')

# target et features
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger le DataFrame nettoyé
df = pd.read_csv('maxcleaned.csv')

# 2. Filtrer les films à faible audience (bottom 20 %)
y = df['box_office_fr']
threshold = y.quantile(0.20)
mask = y >= threshold
df = df[mask].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 3. Créer des poids d’échantillonnage pour favoriser les gros films
sample_weight = np.log1p(y)  # plus de poids aux gros scores

# 4. Split aléatoire train/test
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, sample_weight,
    test_size=0.2, random_state=42
)

# 5. Instancier et entraîner le XGBRegressor avec early stopping
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_test, y_test)],
    sample_weight_eval_set=[w_test],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

# 6. Prédictions
preds = model.predict(X_test)

# 7. Calcul des métriques globales
rmse_all = root_mean_squared_error(y_test, preds)
mae_all  = mean_absolute_error(       y_test, preds)
r2_all   = r2_score(                  y_test, preds)
mape_all = (np.abs((y_test - preds) / y_test).replace([np.inf, -np.inf], np.nan).dropna() * 100).mean()

print("=== Performance globale ===")
print(f"RMSE : {rmse_all:,.0f}")
print(f"MAE  : {mae_all:,.0f}")
print(f"R²   : {r2_all:.3f}")
print(f"MAPE : {mape_all:.1f}%")

# 8. Calcul des métriques sur le top 10 % des films (meilleurs entrants)
q90      = y_test.quantile(0.90)
mask_top = y_test >= q90

rmse_top = root_mean_squared_error(y_test[mask_top], preds[mask_top])
mae_top  = mean_absolute_error(       y_test[mask_top], preds[mask_top])
r2_top   = r2_score(                  y_test[mask_top], preds[mask_top])
mape_top = (np.abs((y_test[mask_top] - preds[mask_top]) / y_test[mask_top])
            .replace([np.inf, -np.inf], np.nan).dropna() * 100).mean()

print("\n=== Performance sur Top 10 % ===")
print(f"RMSE top10% : {rmse_top:,.0f}")
print(f"MAE  top10% : {mae_top:,.0f}")
print(f"R²   top10% : {r2_top:.3f}")
print(f"MAPE top10%: {mape_top:.1f}%")




[0]	validation_0-rmse:318359.20819


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[50]	validation_0-rmse:138018.48572
[100]	validation_0-rmse:132734.69639
[150]	validation_0-rmse:131283.90616
[200]	validation_0-rmse:130783.17766
[240]	validation_0-rmse:130859.42154
=== Performance globale ===
RMSE : 117,487
MAE  : 49,965
R²   : 0.842
MAPE : 75.5%

=== Performance sur Top 10 % ===
RMSE top10% : 310,357
MAE  top10% : 201,816
R²   top10% : 0.677
MAPE top10%: 25.3%


Version 2 du modele 

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger le DataFrame déjà préparé
df = pd.read_csv('maxcleaned.csv')

# 2. Filtrer les films à faible audience (bottom 20 %)
y = df['box_office_fr']
threshold = y.quantile(0.20)
mask = y >= threshold
df = df[mask].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 3. Log-transform de la cible pour stabiliser les valeurs extrêmes
y_log = np.log1p(y)

# 4. Sample weights proportionnels à log1p(y)
w = np.log1p(y)

# 5. Split aléatoire train/test en gardant y et y_log
X_train, X_test, y_train_log, y_test_log, w_train, w_test, y_train, y_test = train_test_split(
    X, y_log, w, y,
    test_size=0.2,
    random_state=42
)

# 6. Instanciation et entraînement du XGBRegressor sur y_log
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(
    X_train, y_train_log,
    sample_weight=w_train,
    eval_set=[(X_test, y_test_log)],
    sample_weight_eval_set=[w_test],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

# 7. Prédictions en espace log, puis back-transform
y_pred_log = model.predict(X_test)
preds = np.expm1(y_pred_log)

# 8. Évaluation globale en espace réel
rmse_all = root_mean_squared_error(y_test, preds)
mae_all  = mean_absolute_error(       y_test, preds)
r2_all   = r2_score(                  y_test, preds)

print("=== Performance globale (après log-transform) ===")
print(f"RMSE : {rmse_all:,.0f}")
print(f"MAE  : {mae_all:,.0f}")
print(f"R²   : {r2_all:.3f}")

# 9. Évaluation sur le top 10 % des films
q90      = y_test.quantile(0.90)
mask_top = y_test >= q90

rmse_top = root_mean_squared_error(y_test[mask_top], preds[mask_top])
mae_top  = mean_absolute_error(       y_test[mask_top], preds[mask_top])
r2_top   = r2_score(                  y_test[mask_top], preds[mask_top])

print("\n=== Performance sur Top 10 % ===")
print(f"RMSE top10% : {rmse_top:,.0f}")
print(f"MAE  top10% : {mae_top:,.0f}")
print(f"R²   top10% : {r2_top:.3f}")


[0]	validation_0-rmse:1.45756


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[50]	validation_0-rmse:0.50313
[100]	validation_0-rmse:0.47921
[150]	validation_0-rmse:0.47745
[186]	validation_0-rmse:0.47767
=== Performance globale (après log-transform) ===
RMSE : 116,706
MAE  : 46,986
R²   : 0.844

=== Performance sur Top 10 % ===
RMSE top10% : 332,468
MAE  top10% : 228,996
R²   top10% : 0.630


V3 

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger le DataFrame préparé
df = pd.read_csv('maxcleaned.csv')

# 2. Filtrer les films à faible audience (bottom 20 %)
y = df['box_office_fr']
threshold = y.quantile(0.20)
mask = y >= threshold
df = df[mask].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 3. Transformer la cible pour stabiliser les blockbusters
y_log = np.log1p(y)

# 4. Poids d’échantillonnage renforçant les gros films
#    Ici on prend la racine carrée pour donner plus de poids aux blockbusters
w = np.sqrt(y)

# 5. Split aléatoire train/test, on conserve y (réel) et y_log (train)
X_train, X_test, y_train_log, y_test_log, w_train, w_test, y_train, y_test = train_test_split(
    X, y_log, w, y,
    test_size=0.2,
    random_state=42
)

# 6. Instancier et entraîner le XGBRegressor sur y_log
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(
    X_train, y_train_log,
    sample_weight=w_train,
    eval_set=[(X_test, y_test_log)],
    sample_weight_eval_set=[w_test],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

# 7. Prédictions (log) et back-transform
y_pred_log = model.predict(X_test)
preds = np.expm1(y_pred_log)

# 8. Évaluation globale en espace réel
rmse_all = root_mean_squared_error(y_test, preds)
mae_all  = mean_absolute_error(       y_test, preds)
r2_all   = r2_score(                  y_test, preds)
mape_all = (np.abs((y_test - preds) / y_test)
            .replace([np.inf, -np.inf], np.nan)
            .dropna().mean() * 100)

print("=== Performance globale ===")
print(f"RMSE  : {rmse_all:,.0f}")
print(f"MAE   : {mae_all:,.0f}")
print(f"R²    : {r2_all:.3f}")
print(f"MAPE  : {mape_all:.1f}%")

# 9. Évaluation sur le top 10 % des films
q90      = y_test.quantile(0.90)
mask_top = y_test >= q90

rmse_top = root_mean_squared_error(y_test[mask_top], preds[mask_top])
mae_top  = mean_absolute_error(       y_test[mask_top], preds[mask_top])
r2_top   = r2_score(                  y_test[mask_top], preds[mask_top])
mape_top = (np.abs((y_test[mask_top] - preds[mask_top]) / y_test[mask_top])
            .replace([np.inf, -np.inf], np.nan)
            .dropna().mean() * 100)

print("\n=== Performance sur Top 10 % ===")
print(f"RMSE top10% : {rmse_top:,.0f}")
print(f"MAE  top10% : {mae_top:,.0f}")
print(f"R²   top10% : {r2_top:.3f}")
print(f"MAPE top10%: {mape_top:.1f}%")


[0]	validation_0-rmse:1.34234
[50]	validation_0-rmse:0.44978


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[100]	validation_0-rmse:0.43082
[150]	validation_0-rmse:0.42868
[182]	validation_0-rmse:0.42938
=== Performance globale ===
RMSE  : 119,369
MAE   : 46,243
R²    : 0.837
MAPE  : 53.1%

=== Performance sur Top 10 % ===
RMSE top10% : 334,133
MAE  top10% : 210,276
R²   top10% : 0.626
MAPE top10%: 25.1%


V4

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger le DataFrame préparé
df = pd.read_csv('maxcleaned.csv')

# 2. Filtrer les films à faible audience (bottom 20 %)
threshold = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= threshold].reset_index(drop=True)

# 3. Séparer la cible et les features
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 4. Créer de nouvelles features d'interaction
X['writer_month_interact']   = X['writer_avg_mean']   * X['month_avg_entries']
X['actor_month_interact']    = X['actor_avg_mean']    * X['month_avg_entries']
X['director_hotmonth']       = X['director_avg_mean'] * X['hot_month']

# 5. Log-transform de la cible
y_log = np.log1p(y)

# 6. Sample weights (accentuation des blockbusters)
w = np.sqrt(y)

# 7. Split aléatoire train/test
X_train, X_test, y_train_log, y_test_log, w_train, w_test, y_train, y_test = train_test_split(
    X, y_log, w, y,
    test_size=0.2,
    random_state=42
)

# 8. Entraînement du XGBRegressor sur y_log
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(
    X_train, y_train_log,
    sample_weight=w_train,
    eval_set=[(X_test, y_test_log)],
    sample_weight_eval_set=[w_test],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

# 9. Prédictions et back-transform
y_pred_log = model.predict(X_test)
preds = np.expm1(y_pred_log)

# 10. Évaluation globale
rmse_all = root_mean_squared_error(y_test, preds)
mae_all  = mean_absolute_error(y_test, preds)
r2_all   = r2_score(y_test, preds)
mape_all = (np.abs((y_test - preds) / y_test)
            .replace([np.inf, -np.inf], np.nan)
            .dropna().mean() * 100)

print("=== Performance globale ===")
print(f"RMSE : {rmse_all:,.0f}")
print(f"MAE  : {mae_all:,.0f}")
print(f"R²   : {r2_all:.3f}")
print(f"MAPE : {mape_all:.1f}%")

# 11. Évaluation sur Top 20 % des films
q80       = y_test.quantile(0.80)
mask_top20 = y_test >= q80

rmse_top20 = root_mean_squared_error(y_test[mask_top20], preds[mask_top20])
mae_top20  = mean_absolute_error(y_test[mask_top20], preds[mask_top20])
r2_top20   = r2_score(y_test[mask_top20], preds[mask_top20])
mape_top20 = (np.abs((y_test[mask_top20] - preds[mask_top20]) / y_test[mask_top20])
              .replace([np.inf, -np.inf], np.nan).dropna().mean() * 100)

print("\n=== Performance sur Top 20 % ===")
print(f"RMSE top20% : {rmse_top20:,.0f}")
print(f"MAE  top20% : {mae_top20:,.0f}")
print(f"R²   top20% : {r2_top20:.3f}")
print(f"MAPE top20%: {mape_top20:.1f}%")

[0]	validation_0-rmse:1.34191
[50]	validation_0-rmse:0.45067


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[100]	validation_0-rmse:0.42953
[150]	validation_0-rmse:0.42726
[200]	validation_0-rmse:0.42711
[230]	validation_0-rmse:0.42739
=== Performance globale ===
RMSE : 113,479
MAE  : 46,090
R²   : 0.853
MAPE : 53.0%

=== Performance sur Top 20 % ===
RMSE top20% : 233,561
MAE  top20% : 140,892
R²   top20% : 0.754
MAPE top20%: 24.8%


V5

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger et filtrer tout comme avant
df = pd.read_csv('maxcleaned.csv')
threshold = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= threshold].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 2. Créer les interactions (déjà validées)
X['writer_month_interact']   = X['writer_avg_mean']   * X['month_avg_entries']
X['actor_month_interact']    = X['actor_avg_mean']    * X['month_avg_entries']
X['director_hotmonth']       = X['director_avg_mean'] * X['hot_month']

# 3. Log-transform de la cible pour l’entraînement
y_log = np.log1p(y)

# 4. Nouvelle pondération : y**0.75
w = np.power(y, 0.75)

# 5. Split train/test (on conserve y_test pour l’évaluation réelle)
X_train, X_test, y_train_log, y_test_log, w_train, w_test, y_train, y_test = train_test_split(
    X, y_log, w, y,
    test_size=0.2, random_state=42
)

# 6. Entraînement XGB sur y_log
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(
    X_train, y_train_log,
    sample_weight=w_train,
    eval_set=[(X_test, y_test_log)],
    sample_weight_eval_set=[w_test],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

# 7. Prédictions & back-transform
y_pred_log = model.predict(X_test)
preds = np.expm1(y_pred_log)

# 8. Évaluation globale
rmse_all = root_mean_squared_error(y_test, preds)
mae_all  = mean_absolute_error(       y_test, preds)
r2_all   = r2_score(                  y_test, preds)
print(f"RMSE global : {rmse_all:,.0f}, MAE : {mae_all:,.0f}, R² : {r2_all:.3f}")

# 9. Évaluation sur le top 20 %
q80       = y_test.quantile(0.80)
mask_top  = y_test >= q80
rmse_top  = root_mean_squared_error(y_test[mask_top], preds[mask_top])
mae_top   = mean_absolute_error(       y_test[mask_top], preds[mask_top])
r2_top    = r2_score(                  y_test[mask_top], preds[mask_top])
print(f"RMSE top20% : {rmse_top:,.0f}, MAE : {mae_top:,.0f}, R² : {r2_top:.3f}")


[0]	validation_0-rmse:1.24213
[50]	validation_0-rmse:0.42103


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[100]	validation_0-rmse:0.40194
[150]	validation_0-rmse:0.40116
[200]	validation_0-rmse:0.40305
[203]	validation_0-rmse:0.40327
RMSE global : 111,227, MAE : 46,469, R² : 0.859
RMSE top20% : 226,295, MAE : 139,559, R² : 0.769


V6

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

# 1. Charger et filtrer bottom 20 %
df = pd.read_csv('maxcleaned.csv')
threshold = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= threshold].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 2. Features d’interactions
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Sample weights = y**0.75
w = np.power(y, 0.75)

# 4. Split train/test
X_train, X_test, w_train, w_test, y_train, y_test = train_test_split(
    X, w, y,
    test_size=0.2, random_state=42
)

# 5. Évaluation top 20 % helper
def eval_top20(y_true, y_pred):
    q = np.quantile(y_true, 0.8)
    mask = y_true >= q
    return root_mean_squared_error(y_true[mask], y_pred[mask])

# 6. Paramètres communs
base_params = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

def train_and_eval(params, label):
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_test, y_test)],
        sample_weight_eval_set=[w_test],
        eval_metric='rmse',
        early_stopping_rounds=50,
        verbose=False
    )
    preds = model.predict(X_test)
    print(f"{label:<20} | RMSE overall: {root_mean_squared_error(y_test, preds):,.0f} "
          f"| RMSE top20%: {eval_top20(y_test.values, preds):,.0f}")

print("=== Comparaison d’objectifs ===")
# a) L2 standard
train_and_eval(
    dict(objective='reg:squarederror', **base_params),
    "L2 (baseline)"
)

# b) Tweedie
train_and_eval(
    dict(objective='reg:tweedie', tweedie_variance_power=1.2, **base_params),
    "Tweedie p=1.2"
)

# c) Quantile (top 80 %)
train_and_eval(
    dict(objective='reg:quantileerror', quantile_alpha=0.8, **base_params),
    "Quantile α=0.8"
)



=== Comparaison d’objectifs ===


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


L2 (baseline)        | RMSE overall: 120,730 | RMSE top20%: 225,874


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Tweedie p=1.2        | RMSE overall: 116,605 | RMSE top20%: 231,014


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Quantile α=0.8       | RMSE overall: 166,934 | RMSE top20%: 270,995


V7

In [10]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

# 1. Charger & filtrer
df = pd.read_csv('maxcleaned.csv')
threshold = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= threshold].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])
# interactions…
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 2. Pondération (à tester en boucle sur alpha)
alpha = 1.0
w = np.power(y, alpha)

# 3. Split
X_train, X_test, w_train, w_test, y_train, y_test = train_test_split(
    X, w, y, test_size=0.2, random_state=42
)

# 4. Fonction d’éval top20
def eval_top20(y_true, y_pred):
    q = np.quantile(y_true, 0.8)
    mask = y_true >= q
    return root_mean_squared_error(y_true[mask], y_pred[mask])

# 5. Paramètres partagés
base_params = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
    early_stopping_rounds=50
)

def train_obj(obj_name, params):
    model = xgb.XGBRegressor(objective=obj_name, **params)
    model.fit(
      X_train, y_train,
      sample_weight=w_train,
      eval_set=[(X_test, y_test)],
      verbose=False
    )
    preds = model.predict(X_test)
    print(f"{obj_name:<20} | RMSE all: {root_mean_squared_error(y_test, preds):,.0f} "
          f"| RMSE top20%: {eval_top20(y_test.values, preds):,.0f}")

print("=== Test des objectifs robustes ===")
train_obj('reg:squarederror',   base_params)
train_obj('reg:absoluteerror',  base_params)
train_obj('reg:pseudohubererror', base_params)


=== Test des objectifs robustes ===
reg:squarederror     | RMSE all: 122,472 | RMSE top20%: 235,912
reg:absoluteerror    | RMSE all: 127,184 | RMSE top20%: 229,572
reg:pseudohubererror | RMSE all: 22,997,599,357,980 | RMSE top20%: 22,997,598,955,635


V7

In [11]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error

# 1. Charger & filtrer le top 80 % comme avant
df = pd.read_csv('maxcleaned.csv')
mask = df['box_office_fr'] >= df['box_office_fr'].quantile(0.20)
df = df[mask].reset_index(drop=True)
y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])
# interactions…
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 2. Poids = y**0.75
w = y**0.75

# 3. Split
X_train, X_test, w_train, w_test, y_train, y_test = train_test_split(
    X, w, y, test_size=0.2, random_state=42
)

# 4. Éval top 20 % helper
def eval_top20(y_true, y_pred):
    q = np.quantile(y_true, 0.8)
    mask = y_true >= q
    return root_mean_squared_error(y_true[mask], y_pred[mask])

# 5. Paramètres communs avec Gamma
params_gamma = {
    'objective': 'reg:gamma',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 1000,
    'random_state': 42,
    'eval_metric': 'rmse',
    'early_stopping_rounds': 50
}

# 6. Entraînement & évaluation
model_gamma = xgb.XGBRegressor(**params_gamma)
model_gamma.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
preds_gamma = model_gamma.predict(X_test)

print("=== Gamma objective ===")
print(f"RMSE all   : {root_mean_squared_error(y_test, preds_gamma):,.0f}")
print(f"RMSE top20%: {eval_top20(y_test.values, preds_gamma):,.0f}")


=== Gamma objective ===
RMSE all   : 123,798
RMSE top20%: 256,035


V7

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger et filtrer (conserver top 80 %)
df = pd.read_csv("maxcleaned.csv")
q20 = df["box_office_fr"].quantile(0.20)
df = df[df["box_office_fr"] >= q20].reset_index(drop=True)

y = df["box_office_fr"]
X = df.drop(columns=["box_office_fr"])

# 2. Features d’interactions
X["writer_month_interact"] = X["writer_avg_mean"] * X["month_avg_entries"]
X["actor_month_interact"]  = X["actor_avg_mean"]  * X["month_avg_entries"]
X["director_hotmonth"]     = X["director_avg_mean"] * X["hot_month"]

# 3. Pondération “step” : 4× pour top 20 %, 1× sinon
cut80   = y.quantile(0.80)
weights = np.where(y >= cut80, 4.0, 1.0)

# 4. Split stratifié par déciles de y
buckets = pd.qcut(y, 10, labels=False, duplicates="drop")
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X, y, weights,
    test_size=0.2,
    random_state=42,
    stratify=buckets,
)

# 5. Custom objective asym L2
def asym_l2(y_true: np.ndarray, y_pred: np.ndarray):
    diff = y_pred - y_true
    # grad = 2 * diff * λ ; hess = 2 * λ
    # λ = 3 si sur-prédiction (diff>0), sinon λ = 1
    lamb = np.where(diff > 0, 3.0, 1.0)
    grad = 2.0 * lamb * diff
    hess = 2.0 * lamb
    return grad, hess

# 6. Paramètres XGBoost (sans clé "objective")
params = {
    "learning_rate":        0.05,
    "max_depth":            7,
    "subsample":            0.8,
    "colsample_bytree":     0.8,
    "n_estimators":         3000,
    "random_state":         42,
    "eval_metric":          "rmse",
    "early_stopping_rounds":100,
}

# 7. Entraîner avec objectif custom
model = xgb.XGBRegressor(objective=asym_l2, **params)
model.fit(
    X_tr, y_tr,
    sample_weight=w_tr,
    eval_set=[(X_te, y_te)],
    sample_weight_eval_set=[w_te],
    verbose=200,
)

# 8. Évaluer
preds = model.predict(X_te)

def print_metrics(y_true, y_pred, label):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(     y_true, y_pred)
    r2   = r2_score(                y_true, y_pred)
    print(f"\n=== {label} ===")
    print(f"RMSE : {rmse:,.0f}")
    print(f"MAE  : {mae:,.0f}")
    print(f"R²   : {r2:.3f}")

print_metrics(y_te, preds, "GLOBAL")
mask_top20 = y_te >= cut80
print_metrics(y_te[mask_top20], preds[mask_top20], "TOP 20 %")




[0]	validation_0-rmse:547373.58610
[200]	validation_0-rmse:193039.31892
[400]	validation_0-rmse:191974.38394
[600]	validation_0-rmse:191424.04883
[800]	validation_0-rmse:191070.47761
[1000]	validation_0-rmse:190919.03211
[1200]	validation_0-rmse:190822.42857
[1400]	validation_0-rmse:190749.89211
[1600]	validation_0-rmse:190716.05806
[1800]	validation_0-rmse:190687.63758
[2000]	validation_0-rmse:190670.57351
[2200]	validation_0-rmse:190654.20633
[2400]	validation_0-rmse:190646.51329
[2600]	validation_0-rmse:190639.87543
[2800]	validation_0-rmse:190637.00821
[2999]	validation_0-rmse:190635.92061

=== GLOBAL ===
RMSE : 128,675
MAE  : 57,972
R²   : 0.841

=== TOP 20 % ===
RMSE : 263,279
MAE  : 171,558
R²   : 0.723


double modele 

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    roc_auc_score
)

# 1. Charger et filtrer bottom 20 %
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

# 2. Features existantes + interactions
X = df.drop(columns=['box_office_fr'])
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Label de classification : 1 si top 20 %, sinon 0
q80 = df['box_office_fr'].quantile(0.80)
y_class = (df['box_office_fr'] >= q80).astype(int)
y_reg   = df['box_office_fr']  # pour la régression

# 4. Split train/test (stratifié sur y_class pour garder la proportion de blockbusters)
X_tr, X_te, yc_tr, yc_te, y_tr, y_te = train_test_split(
    X, y_class, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

# 5. Entraîner le classifieur
clf = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    use_label_encoder=False
)
clf.fit(
    X_tr, yc_tr,
    eval_set=[(X_te, yc_te)],
    early_stopping_rounds=20,
    verbose=False
)
# Score classification
probs = clf.predict_proba(X_te)[:,1]
print("ROC AUC classifier :", roc_auc_score(yc_te, probs))

# 6. Construire deux jeux pour la régression
mask_tr_top = yc_tr == 1
mask_te_top = yc_te == 1

X_tr_base, y_tr_base = X_tr[~mask_tr_top], y_tr[~mask_tr_top]
X_tr_top , y_tr_top  = X_tr[ mask_tr_top], y_tr[ mask_tr_top]

X_te_base, y_te_base = X_te[~mask_te_top], y_te[~mask_te_top]
X_te_top , y_te_top  = X_te[ mask_te_top], y_te[ mask_te_top]

# Poids pour la régression (on peut garder y**0.75 ou log-transform si besoin)
w_tr_base = np.power(y_tr_base, 0.75)
w_tr_top  = np.power(y_tr_top , 0.75)

# 7. Entraîner le régresseur « non-top » (L2 simple)
reg_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
    early_stopping_rounds=30
)
reg_base.fit(
    X_tr_base, y_tr_base,
    sample_weight=w_tr_base,
    eval_set=[(X_te_base, y_te_base)],
    verbose=False
)

# 8. Entraîner le régresseur « top20 » (L2 ou Tweedie si tu préfères)
reg_top = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
    early_stopping_rounds=30
)
reg_top.fit(
    X_tr_top, y_tr_top,
    sample_weight=w_tr_top,
    eval_set=[(X_te_top, y_te_top)],
    verbose=False
)

# 9. Inférence & évaluation
# a) Prédiction de la probabilité d’être blockbuster
is_top_pred = (clf.predict_proba(X_te)[:,1] >= 0.5)

# b) Pour chaque film, on choisit le modèle correspondant
preds = np.where(
    is_top_pred,
    reg_top.predict(X_te),
    reg_base.predict(X_te)
)

# c) Metrics globales
def print_metrics(y_true, y_pred, label):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(     y_true, y_pred)
    r2   = r2_score(                y_true, y_pred)
    print(f"\n=== {label} ===")
    print(f"RMSE : {rmse:,.0f}")
    print(f"MAE  : {mae:,.0f}")
    print(f"R²   : {r2:.3f}")

from sklearn.metrics import mean_absolute_error
print_metrics(y_te, preds, "GLOBAL")

# d) Metrics sur top 20 % « réels »
mask_true_top = y_te >= q80
print_metrics(y_te[mask_true_top], preds[mask_true_top], "TOP 20 %")


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


ROC AUC classifier : 0.9761059623813001

=== GLOBAL ===
RMSE : 127,023
MAE  : 58,609
R²   : 0.819

=== TOP 20 % ===
RMSE : 240,056
MAE  : 177,043
R²   : 0.714


Version blockbuster sans optuna et cross.

In [17]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Charger et filtrer (garder top 60 %)
df = pd.read_csv('maxcleaned.csv')
df = df[df.box_office_fr >= df.box_office_fr.quantile(0.40)].reset_index(drop=True)

y = df.box_office_fr
X = df.drop(columns=['box_office_fr'])

# 2. Features d’interaction
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Pondération “step” : 1× / 4× / 10×
q80, q95 = y.quantile([0.8, 0.95])
w = np.where(y >= q95, 10.0,
     np.where(y >= q80, 4.0, 1.0))

# 4. Split stratifié (déciles) pour garder la proportion de blockbusters
buckets = pd.qcut(y, 10, labels=False, duplicates='drop')
Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
    X, y, w,
    test_size=0.2,
    random_state=42,
    stratify=buckets
)

# 5. Custom loss asymétrique L2 (sur-prédiction ×3)
def asym_l2(y_true, y_pred):
    diff = y_pred - y_true
    lam  = np.where(diff > 0, 3.0, 1.0)
    grad = 2.0 * lam * diff
    hess = 2.0 * lam
    return grad, hess

# 6. Entraînement
model = xgb.XGBRegressor(
    objective=asym_l2,
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    early_stopping_rounds=100,
    n_jobs=-1,
    random_state=42
)
model.fit(
    Xtr, ytr,
    sample_weight=wtr,
    eval_set=[(Xte, yte)],
    sample_weight_eval_set=[wte],
    verbose=100
)

# 7. Évaluation
pred = model.predict(Xte)

def report(y_true, y_pred, tag):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"[{tag}] RMSE {rmse:,.0f} | MAE {mae:,.0f} | R² {r2:.3f}")

report(yte,      pred,          "GLOBAL")
report(yte[yte>=q80], pred[yte>=q80], "TOP 20%")



[0]	validation_0-rmse:764836.59818
[100]	validation_0-rmse:276483.99466
[200]	validation_0-rmse:266784.87382
[300]	validation_0-rmse:263805.29680
[400]	validation_0-rmse:262118.67199
[500]	validation_0-rmse:261618.52266
[600]	validation_0-rmse:261616.93347
[622]	validation_0-rmse:261632.88473
[GLOBAL] RMSE 150,164 | MAE 71,009 | R² 0.788
[TOP 20%] RMSE 275,649 | MAE 188,550 | R² 0.661


In [18]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Chargement & filtrage : on garde les 80 % de films les plus forts
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 2. Interactions (inchangées)
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Pondération step (1× / 3× / 6×)
q80, q95 = y.quantile([0.8, 0.95])
w = np.where(y >= q95, 6.0,
     np.where(y >= q80, 3.0, 1.0))

# 4. Split stratifié par déciles pour garder la proportion de blockbusters
buckets = pd.qcut(y, 10, labels=False, duplicates='drop')
Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
    X, y, w,
    test_size=0.2,
    random_state=42,
    stratify=buckets
)

# 5. Entraînement XGBoost with Tweedie (p=1.2)
model = xgb.XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1.2,
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42
)
model.fit(
    Xtr, ytr,
    sample_weight=wtr,
    eval_set=[(Xte, yte)],
    sample_weight_eval_set=[wte],
    verbose=100
)

# 6. Évaluation
pred = model.predict(Xte)

def report(y_true, y_pred, name):
    print(f"\n[{name}]")
    print(" RMSE :", f"{root_mean_squared_error(y_true, y_pred):,.0f}")
    print(" MAE  :", f"{mean_absolute_error(y_true, y_pred):,.0f}")
    print(" R²   :", f"{r2_score(y_true, y_pred):.3f}")

report(yte,          pred,     "GLOBAL")
report(yte[yte>=q80], pred[yte>=q80], "TOP 20 %")


[0]	validation_0-rmse:674174.02121
[100]	validation_0-rmse:212264.15086
[200]	validation_0-rmse:212742.22405
[201]	validation_0-rmse:212796.98522

[GLOBAL]
 RMSE : 121,957
 MAE  : 55,795
 R²   : 0.857

[TOP 20 %]
 RMSE : 246,225
 MAE  : 164,290
 R²   : 0.758


In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score
)

# 1. Chargement & filtrage bottom 20 %
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

y = df['box_office_fr']
X = df.drop(columns=['box_office_fr'])

# 2. Features d’interaction (inchangées)
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Pondération step plus marquée (1× / 5× / 12×)
q80, q95 = y.quantile([0.8, 0.95])
w = np.where(y >= q95, 12.0,
     np.where(y >= q80, 5.0, 1.0))

# 4. Split stratifié pour garder la même proportion de gros films
buckets = pd.qcut(y, 10, labels=False, duplicates='drop')
Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
    X, y, w,
    test_size=0.2,
    random_state=42,
    stratify=buckets
)

# 5. Modèle XGBoost en Quantile 0.80
model = xgb.XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=0.8,          # on optimise le 80ᵉ quantile
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42
)

model.fit(
    Xtr, ytr,
    sample_weight=wtr,
    eval_set=[(Xte, yte)],
    sample_weight_eval_set=[wte],
    verbose=100
)

# 6. Évaluation
pred = model.predict(Xte)

def report(y_true, y_pred, tag):
    print(f"\n[{tag}]")
    print(" RMSE :", f"{root_mean_squared_error(y_true, y_pred):,.0f}")
    print(" MAE  :", f"{mean_absolute_error(y_true, y_pred):,.0f}")
    print(" R²   :", f"{r2_score(y_true, y_pred):.3f}")

report(yte,          pred,      "GLOBAL")
report(yte[yte>=q80], pred[yte>=q80], "TOP 20 %")


[0]	validation_0-rmse:659166.44959
[100]	validation_0-rmse:263218.61354
[200]	validation_0-rmse:256962.12319
[300]	validation_0-rmse:252873.69064
[400]	validation_0-rmse:251427.50627
[475]	validation_0-rmse:251053.24389

[GLOBAL]
 RMSE : 165,752
 MAE  : 97,620
 R²   : 0.737

[TOP 20 %]
 RMSE : 264,106
 MAE  : 190,220
 R²   : 0.721


best version tentative amelioration 

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
    roc_auc_score
)

# 1. Chargement + filtre bottom 20 %
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

# 2. Préparation des features
X = df.drop(columns=['box_office_fr'])
X['writer_month_interact'] = X['writer_avg_mean'] * X['month_avg_entries']
X['actor_month_interact']  = X['actor_avg_mean']  * X['month_avg_entries']
X['director_hotmonth']     = X['director_avg_mean'] * X['hot_month']

# 3. Labels
y = df['box_office_fr']
q80 = y.quantile(0.80)
y_class = (y >= q80).astype(int)

# 4. Split train/test stratifié
X_tr, X_te, yc_tr, yc_te, y_tr, y_te = train_test_split(
    X, y_class, y,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

# 5. Classifieur XGB
clf = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    use_label_encoder=False
)
clf.fit(
    X_tr, yc_tr,
    eval_set=[(X_te, yc_te)],
    early_stopping_rounds=10,
    verbose=False
)
probs = clf.predict_proba(X_te)[:,1]
print("ROC AUC classifier :", roc_auc_score(yc_te, probs))

# 6. Préparation des DFs pour régression
mask_tr_top = yc_tr == 1
mask_te_top = yc_te == 1

Xb_tr, yb_tr = X_tr[~mask_tr_top], y_tr[~mask_tr_top]
Xt_tr, yt_tr = X_tr[ mask_tr_top], y_tr[ mask_tr_top]
Xb_te, yb_te = X_te[~mask_te_top], y_te[~mask_te_top]
Xt_te, yt_te = X_te[ mask_te_top], y_te[ mask_te_top]

# 7. Log-transform & poids pour chaque régresseur
yb_tr_log = np.log1p(yb_tr)
yt_tr_log = np.log1p(yt_tr)

w_b = np.sqrt(yb_tr)        # base : sqrt
w_t = np.power(yt_tr, 0.8)   # top : y**0.8

# 8. Régresseur « base » (log-space L2)
reg_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=20
)
reg_base.fit(
    Xb_tr, yb_tr_log,
    sample_weight=w_b,
    eval_set=[(Xb_te, np.log1p(yb_te))],
    verbose=False
)

# 9. Régresseur « top20 » (log-space Tweedie)
reg_top = xgb.XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1.2,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=20
)
reg_top.fit(
    Xt_tr, yt_tr_log,
    sample_weight=w_t,
    eval_set=[(Xt_te, np.log1p(yt_te))],
    verbose=False
)

# 10. Inférence & blending soft
#    on « renforce » la proba haute par exponentiation
alpha = 0.7
p = probs ** alpha
pred_base = np.expm1(reg_base.predict(X_te))
pred_top  = np.expm1(reg_top.predict(X_te))
preds = pred_base * (1 - p) + pred_top * p

# 11. Évaluation
def print_metrics(y_true, y_pred, tag):
    print(f"\n=== {tag} ===")
    print("RMSE :", f"{root_mean_squared_error(y_true, y_pred):,.0f}")
    print("MAE  :", f"{mean_absolute_error(y_true, y_pred):,.0f}")
    print("R²   :", f"{r2_score(y_true, y_pred):.3f}")

print_metrics(y_te, preds, "GLOBAL")
print_metrics(y_te[y_te>=q80], preds[y_te>=q80], "TOP 20 %")


/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


ROC AUC classifier : 0.9757464390065742

=== GLOBAL ===
RMSE : 124,154
MAE  : 54,748
R²   : 0.827

=== TOP 20 % ===
RMSE : 245,480
MAE  : 162,057
R²   : 0.700


version final avec stratified et optuna 

In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import joblib
import pickle

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error, roc_auc_score

# 1. Charger & filtrer bottom 20 %
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

# 2. Construire X, y_class, y_reg + interactions
y_reg   = df['box_office_fr'].values
q80     = np.quantile(y_reg, 0.80)
y_class = (y_reg >= q80).astype(int)

X_df = df.drop(columns=['box_office_fr'])
X_df['writer_month_interact'] = X_df['writer_avg_mean'] * X_df['month_avg_entries']
X_df['actor_month_interact']  = X_df['actor_avg_mean']  * X_df['month_avg_entries']
X_df['director_hotmonth']     = X_df['director_avg_mean'] * X_df['hot_month']
X = X_df.values

# Déciles pour stratification
bucket = np.digitize(y_reg, np.quantile(y_reg, np.linspace(0,1,11))[1:-1])

# helper RMSE
def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

# 3. Fit & score par fold
def fit_and_score(X_tr, yc_tr, y_tr, X_va, yc_va, y_va, gamma, lam):
    # 3.1 Classifieur CPU
    clf = xgb.XGBClassifier(
        tree_method='hist', n_jobs=-1,
        n_estimators=100, learning_rate=0.1, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='auc', use_label_encoder=False,
        random_state=42, early_stopping_rounds=10
    )
    clf.fit(X_tr, yc_tr, eval_set=[(X_va, yc_va)], verbose=False)
    p_va = clf.predict_proba(X_va)[:,1]

    # 3.2 Séparer base vs top
    mask_tr_top = yc_tr == 1
    mask_va_top = yc_va == 1
    Xb_tr, yb_tr = X_tr[~mask_tr_top], y_tr[~mask_tr_top]
    Xt_tr, yt_tr = X_tr[ mask_tr_top], y_tr[ mask_tr_top]
    Xb_va, yb_va = X_va[~mask_va_top], y_va[~mask_va_top]
    Xt_va, yt_va = X_va[ mask_va_top], y_va[ mask_va_top]

    # 3.3 Régresseur base (L2)
    reg_b = xgb.XGBRegressor(
        tree_method='hist', n_jobs=-1,
        objective='reg:squarederror',
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='rmse', random_state=42,
        early_stopping_rounds=20
    )
    w_b = np.sqrt(yb_tr)
    reg_b.fit(Xb_tr, yb_tr, sample_weight=w_b, eval_set=[(Xb_va, yb_va)], verbose=False)
    yb_pred = reg_b.predict(X_va)

    # 3.4 Régresseur top (L1)
    reg_t = xgb.XGBRegressor(
        tree_method='hist', n_jobs=-1,
        objective='reg:absoluteerror',
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='rmse', random_state=42,
        early_stopping_rounds=20
    )
    w_t = np.power(yt_tr, gamma)
    reg_t.fit(Xt_tr, yt_tr, sample_weight=w_t, eval_set=[(Xt_va, yt_va)], verbose=False)
    yt_pred = reg_t.predict(X_va)

    # 3.5 Soft-blend
    p = p_va ** lam
    blend = yb_pred * (1 - p) + yt_pred * p

    # 3.6 RMSE top 20 %
    return rmse(yt_va, blend[mask_va_top])

# 4. Objectif Optuna (γ, λ)
def objective(trial):
    gamma = trial.suggest_float('gamma', 0.5, 1.5)
    lam   = trial.suggest_float('lambda_soft', 0.5, 1.5)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    vals = []
    for tr_idx, va_idx in skf.split(X, bucket):
        vals.append(fit_and_score(
            X[tr_idx], y_class[tr_idx], y_reg[tr_idx],
            X[va_idx], y_class[va_idx], y_reg[va_idx],
            gamma, lam
        ))
    return np.mean(vals)

# 5. Lancer Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, timeout=1200)
best = study.best_params
print("→ Meilleurs params :", best, "→ CV RMSE :", study.best_value)

# 6. Entraînement final
gamma, lam = best['gamma'], best['lambda_soft']

clf_final = xgb.XGBClassifier(
    tree_method='hist', n_jobs=-1,
    n_estimators=100, learning_rate=0.1, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', use_label_encoder=False,
    random_state=42
)
clf_final.fit(X, y_class, verbose=False)

reg_base_final = xgb.XGBRegressor(
    tree_method='hist', n_jobs=-1,
    objective='reg:squarederror',
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', random_state=42
)
reg_base_final.fit(X, y_reg, sample_weight=np.sqrt(y_reg), verbose=False)

mask_top = y_class == 1
reg_top_final = xgb.XGBRegressor(
    tree_method='hist', n_jobs=-1,
    objective='reg:absoluteerror',
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', random_state=42
)
reg_top_final.fit(
    X[mask_top], y_reg[mask_top],
    sample_weight=np.power(y_reg[mask_top], gamma),
    verbose=False
)

# 7. Sérialisation sous xgboostmax
ensemble = {
    'classifier':     clf_final,
    'regressor_base': reg_base_final,
    'regressor_top':  reg_top_final,
    'gamma':          gamma,
    'lambda_soft':    lam
}
joblib.dump(ensemble, 'xgboostmax.joblib')
with open('xgboostmax.pkl','wb') as f:
    pickle.dump(ensemble, f)

print("✅ xgboostmax.joblib & xgboostmax.pkl créés.")






[I 2025-04-24 01:31:50,635] A new study created in memory with name: no-name-29fb490f-0954-462f-a5ce-d79b215200c7
[I 2025-04-24 01:31:53,690] Trial 0 finished with value: 265014.1352013941 and parameters: {'gamma': 0.9393075545669357, 'lambda_soft': 1.175940603826894}. Best is trial 0 with value: 265014.1352013941.
[I 2025-04-24 01:31:56,829] Trial 1 finished with value: 272630.30608808214 and parameters: {'gamma': 0.6896568667001706, 'lambda_soft': 1.459115386413119}. Best is trial 0 with value: 265014.1352013941.
[I 2025-04-24 01:32:00,476] Trial 2 finished with value: 269149.3845017378 and parameters: {'gamma': 1.2148567929302803, 'lambda_soft': 1.216774290195687}. Best is trial 0 with value: 265014.1352013941.
[I 2025-04-24 01:32:03,186] Trial 3 finished with value: 269134.57135784865 and parameters: {'gamma': 1.2701822542675036, 'lambda_soft': 1.3668169960420116}. Best is trial 0 with value: 265014.1352013941.
[I 2025-04-24 01:32:06,170] Trial 4 finished with value: 261567.0085454

→ Meilleurs params : {'gamma': 0.8510076111450886, 'lambda_soft': 0.501476025951975} → CV RMSE : 252250.82108115026
✅ xgboostmax.joblib & xgboostmax.pkl créés.


V2 avec optuna 

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import joblib
import pickle

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error, roc_auc_score

# Helper pour le RMSE
def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

# 1. Charger & filtrer bottom 20 %
df = pd.read_csv('maxcleaned.csv')
q20 = df['box_office_fr'].quantile(0.20)
df = df[df['box_office_fr'] >= q20].reset_index(drop=True)

# 2. Préparer X, y_class, y_reg + interactions
y_reg   = df['box_office_fr'].values
q80     = np.quantile(y_reg, 0.80)
y_class = (y_reg >= q80).astype(int)

X_df = df.drop(columns=['box_office_fr'])
X_df['writer_month_interact'] = X_df['writer_avg_mean'] * X_df['month_avg_entries']
X_df['actor_month_interact']  = X_df['actor_avg_mean']  * X_df['month_avg_entries']
X_df['director_hotmonth']     = X_df['director_avg_mean'] * X_df['hot_month']
X = X_df.values

# Déciles pour stratification
bucket = np.digitize(
    y_reg,
    np.quantile(y_reg, np.linspace(0,1,11))[1:-1]
)

# 3. Fonction fit & score pour Optuna (moyenne sur 5 folds)
def fit_and_score(trial):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score_folds = []
    
    for tr_idx, va_idx in skf.split(X, bucket):
        X_tr, X_va = X[tr_idx], X[va_idx]
        yc_tr, yc_va = y_class[tr_idx], y_class[va_idx]
        y_tr, y_va   = y_reg[tr_idx],       y_reg[va_idx]

        # 3.1 Classifieur CPU
        clf = xgb.XGBClassifier(
            **{
                'tree_method':'hist',
                'n_jobs':-1,
                'n_estimators': trial.suggest_int('clf_n_estimators', 50, 200),
                'learning_rate': trial.suggest_loguniform('clf_lr', 0.01, 0.3),
                'max_depth': trial.suggest_int('clf_max_depth', 3, 8),
                'subsample': trial.suggest_float('clf_sub', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('clf_col', 0.6, 1.0),
                'eval_metric':'auc',
                'use_label_encoder':False,
                'random_state':42,
                'early_stopping_rounds':10
            }
        )
        clf.fit(X_tr, yc_tr, eval_set=[(X_va, yc_va)], verbose=False)
        p_va = clf.predict_proba(X_va)[:,1]

        # 3.2 Séparer base vs top
        mask_tr_top = yc_tr == 1
        mask_va_top = yc_va == 1

        Xb_tr, yb_tr = X_tr[~mask_tr_top], y_tr[~mask_tr_top]
        Xt_tr, yt_tr = X_tr[ mask_tr_top], y_tr[ mask_tr_top]
        Xb_va, yb_va = X_va[~mask_va_top], y_va[~mask_va_top]
        Xt_va, yt_va = X_va[ mask_va_top], y_va[ mask_va_top]

        # 3.3 Base regressor
        reg_b = xgb.XGBRegressor(
            **{
                'tree_method':'hist',
                'n_jobs':-1,
                'objective':'reg:squarederror',
                'n_estimators': trial.suggest_int('base_n_estimators', 100, 400),
                'learning_rate': trial.suggest_loguniform('base_lr', 0.01, 0.1),
                'max_depth': trial.suggest_int('base_max_depth', 4, 10),
                'subsample': trial.suggest_float('base_sub', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('base_col', 0.6, 1.0),
                'eval_metric':'rmse',
                'random_state':42,
                'early_stopping_rounds':20
            }
        )
        w_b = np.sqrt(yb_tr)
        reg_b.fit(Xb_tr, yb_tr, sample_weight=w_b, eval_set=[(Xb_va, yb_va)], verbose=False)
        yb_pred = reg_b.predict(X_va)

        # 3.4 Top regressor
        top_obj = trial.suggest_categorical('top_obj', ['reg:absoluteerror','reg:tweedie','reg:gamma'])
        top_params = {
            'tree_method':'hist',
            'n_jobs':-1,
            'objective': top_obj,
            'n_estimators': trial.suggest_int('top_n_estimators', 100, 400),
            'learning_rate': trial.suggest_loguniform('top_lr', 0.01, 0.1),
            'max_depth': trial.suggest_int('top_max_depth', 4, 10),
            'subsample': trial.suggest_float('top_sub', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('top_col', 0.6, 1.0),
            'eval_metric':'rmse',
            'random_state':42,
            'early_stopping_rounds':20
        }
        if top_obj == 'reg:tweedie':
            top_params['tweedie_variance_power'] = trial.suggest_float('tweedie_p', 1.0, 1.8)
        if top_obj == 'reg:gamma':
            top_params['gamma'] = trial.suggest_loguniform('reg_gamma', 1e-3, 10.0)

        reg_t = xgb.XGBRegressor(**top_params)
        power_y = trial.suggest_float('power_y', 0.5, 1.5)
        w_t = np.power(yt_tr, power_y)
        reg_t.fit(Xt_tr, yt_tr, sample_weight=w_t, eval_set=[(Xt_va, yt_va)], verbose=False)
        yt_pred = reg_t.predict(X_va)

        # 3.5 Soft-blend
        lam = trial.suggest_float('lambda_soft', 0.3, 1.0)
        p = np.power(p_va, lam)
        blend = yb_pred * (1 - p) + yt_pred * p

        # 3.6 Calcul des métriques
        rmse_all = rmse(y_va, blend)
        rmse_top = rmse(yt_va, blend[mask_va_top])

        # objectif mixte : 10× top20 + global
        score_folds.append(10 * rmse_top + rmse_all)

    return np.mean(score_folds)

# 4. Étude Optuna
study = optuna.create_study(direction='minimize')
study.optimize(fit_and_score, n_trials=50, timeout=1800)

print("▶ Best params:", study.best_params)
print("▶ Best objective:", study.best_value)

# 5. Entraînement final sur tout le jeu
best = study.best_params

# Classifieur final
clf_final = xgb.XGBClassifier(
    **{
        'tree_method':'hist',
        'n_jobs':-1,
        'n_estimators': best['clf_n_estimators'],
        'learning_rate': best['clf_lr'],
        'max_depth': best['clf_max_depth'],
        'subsample': best['clf_sub'],
        'colsample_bytree': best['clf_col'],
        'eval_metric':'auc',
        'use_label_encoder':False,
        'random_state':42
    }
)
clf_final.fit(X, y_class, verbose=False)
print("ROC AUC (train) :", roc_auc_score(y_class, clf_final.predict_proba(X)[:,1]))

# Base regressor final
reg_b_final = xgb.XGBRegressor(
    **{
        'tree_method':'hist',
        'n_jobs':-1,
        'objective':'reg:squarederror',
        'n_estimators': best['base_n_estimators'],
        'learning_rate': best['base_lr'],
        'max_depth': best['base_max_depth'],
        'subsample': best['base_sub'],
        'colsample_bytree': best['base_col'],
        'eval_metric':'rmse',
        'random_state':42
    }
)
reg_b_final.fit(X, y_reg, sample_weight=np.sqrt(y_reg), verbose=False)

# Top regressor final
top_obj = best['top_obj']
top_params = {
    'tree_method':'hist',
    'n_jobs':-1,
    'objective': top_obj,
    'n_estimators': best['top_n_estimators'],
    'learning_rate': best['top_lr'],
    'max_depth': best['top_max_depth'],
    'subsample': best['top_sub'],
    'colsample_bytree': best['top_col'],
    'eval_metric':'rmse',
    'random_state':42
}
if top_obj == 'reg:tweedie':
    top_params['tweedie_variance_power'] = best['tweedie_p']
if top_obj == 'reg:gamma':
    top_params['gamma'] = best['reg_gamma']

reg_t_final = xgb.XGBRegressor(**top_params)
mask_top = y_class == 1
w_t_final = np.power(y_reg[mask_top], best['power_y'])
reg_t_final.fit(X[mask_top], y_reg[mask_top], sample_weight=w_t_final, verbose=False)

# 6. Sérialisation
ensemble = {
    'classifier':     clf_final,
    'regressor_base': reg_b_final,
    'regressor_top':  reg_t_final,
    'params':         best
}
joblib.dump(ensemble, 'xgboostmax.joblib')
with open('xgboostmax.pkl','wb') as f:
    pickle.dump(ensemble, f)

print("✅ Ensemble serialisé sous xgboostmax.joblib & xgboostmax.pkl")


[I 2025-04-24 02:02:37,634] A new study created in memory with name: no-name-4fc5dd31-6b03-4e2c-893f-7dafcc3e4237
/tmp/ipykernel_66055/3515129118.py:53: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('clf_lr', 0.01, 0.3),
/tmp/ipykernel_66055/3515129118.py:82: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('base_lr', 0.01, 0.1),
/tmp/ipykernel_66055/3515129118.py:102: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': t

▶ Best params: {'clf_n_estimators': 57, 'clf_lr': 0.2030180030275399, 'clf_max_depth': 4, 'clf_sub': 0.6448785277187968, 'clf_col': 0.7921224937303244, 'base_n_estimators': 262, 'base_lr': 0.01689957909892115, 'base_max_depth': 7, 'base_sub': 0.6231540129270209, 'base_col': 0.8394165712209517, 'top_obj': 'reg:absoluteerror', 'top_n_estimators': 211, 'top_lr': 0.042431488715756464, 'top_max_depth': 5, 'top_sub': 0.7234971281210377, 'top_col': 0.8164638596080382, 'power_y': 0.7540460585177118, 'lambda_soft': 0.4319971329661817}
▶ Best objective: 2653041.8578216047
ROC AUC (train) : 0.9929992238860482
✅ Ensemble serialisé sous xgboostmax.joblib & xgboostmax.pkl
